# Initialization Study — Stage 1

This notebook answers the first research question: how much classification
performance is surrendered by replacing Gaussian weight initialization with
generators an edge device can evaluate in native integer arithmetic.

The comparison is made on a **frozen** reservoir. Only the readout is trained, so
that no part of the difference between initializers can be attributed to
subsequent adaptation of the recurrent weights: what is measured is the quality
of the random projection itself, which is precisely the quantity the
initialization method determines. Stage 2 in Notebook 05 then asks how much of
any deficit training recovers.

Five methods are compared, all injected into an identical substrate at $k = T$
with the configuration selected in Notebook 03:

- **Gaussian** — the baseline, requiring a Box–Muller or inverse-CDF transform
- **LCG** and **LFSR** — pseudo-random generators built from an integer
  multiply-add and a shift-and-XOR respectively
- **Sobol** and **Halton** — deterministic low-discrepancy sequences

Every method is scaled to the same weight variance, $1/f_\text{eff}$. This is an
experimental control rather than a convenience: a uniform variate on $[-1,1]$ has
standard deviation $1/\sqrt{3}$ against the Gaussian's $1$, so without the
correction the baseline would carry weights $1.73$ times larger and any
difference would confound the *shape* of the distribution with its *scale*. The
final section reproduces that confound deliberately, as an ablation.

In [1]:
import pandas as pd
import torch

from edgesnn.data_utils import make_datasets
from edgesnn.experiments import (
    CONNECTIVITY, INIT_METHODS, N_RES, load_config, run_grid, seeds_for,
)
from edgesnn.initializers import initialize_reservoir, is_deterministic
from edgesnn.plotting import plot_initializer_effect

DATA_DIR = "../Data"
FIG_DIR = "../reports/figures"
DATASETS = ["ECG5000", "Wafer", "InsectSound"]
N_SEEDS = 5

splits = {name: make_datasets(DATA_DIR, name) for name in DATASETS}
for method in INIT_METHODS:
    print(f"{method:<10} seeds={seeds_for(method, N_SEEDS)}"
          f"{'   (sequence is deterministic; mask and fit are not)' if is_deterministic(method) else ''}")

gaussian   seeds=[0, 1, 2, 3, 4]
lcg        seeds=[0, 1, 2, 3, 4]
lfsr       seeds=[0, 1, 2, 3, 4]
sobol      seeds=[0, 1, 2, 3, 4]   (sequence is deterministic; mask and fit are not)
halton     seeds=[0, 1, 2, 3, 4]   (sequence is deterministic; mask and fit are not)


## 1. Verifying the variance control


Before comparing the methods it is worth confirming that the control does what
it claims. Each generator is drawn at the reservoir's dimensions and its
empirical standard deviation compared against the target $1/\sqrt{f_\text{eff}}$,
where the effective fan-in counts only the synapses that exist.

If the five agree, then any difference in the results that follow is a
consequence of how the weights are *distributed*, not of how large they are.

In [2]:
target = (CONNECTIVITY * N_RES) ** -0.5
rows = []
for method in INIT_METHODS:
    weights = initialize_reservoir((N_RES, N_RES), method, fan_in=N_RES,
                                   connectivity=CONNECTIVITY, seed=0)
    active = weights[weights != 0]
    rows.append({"method": method, "std": active.std().item(),
                 "target": target,
                 "deviation": active.std().item() / target - 1,
                 "non_zeros": int((weights != 0).sum())})
display(pd.DataFrame(rows).round(4))

,method,std,target,deviation,non_zeros
0,gaussian,0.1962,0.1976,-0.0075,6554
1,lcg,0.1964,0.1976,-0.0064,6554
2,lfsr,0.1980,0.1976,0.0016,6554
3,sobol,0.1946,0.1976,-0.0153,6554
4,halton,0.1972,0.1976,-0.0023,6554


The non-zero count is identical across methods by construction: the connectivity
mask fixes an exact number of synapses rather than drawing each entry
independently, so every initializer operates under the same synaptic budget. That
matters for the energy comparison, since the synapse count is exactly the term
converting spikes into synaptic operations.

## 2. Determinism of the low-discrepancy sequences

Sobol and Halton are constructions, not samplers: the $n$-th point is a fixed
function of $n$, so the seed has nothing to act upon. The check below confirms
this holds in the implementation, because it determines how the two are run —
once each, with no error bar — and reporting a standard deviation of zero is only
honest if it reflects the mathematics rather than an oversight.

In [3]:
# The determinism of a low-discrepancy sequence is a property of the SEQUENCE,
# not of the initialized reservoir. Comparing the dense draw shows the former;
# comparing the matrix the model actually receives shows the latter. They differ,
# and conflating them is what led an earlier version of this study to run Sobol
# and Halton at a single seed and understate them.
for method in INIT_METHODS:
    dense_a = initialize_reservoir((N_RES, N_RES), method, fan_in=N_RES, seed=0)
    dense_b = initialize_reservoir((N_RES, N_RES), method, fan_in=N_RES, seed=99)
    sparse_a = initialize_reservoir((N_RES, N_RES), method, fan_in=N_RES,
                                    connectivity=CONNECTIVITY, seed=0)
    sparse_b = initialize_reservoir((N_RES, N_RES), method, fan_in=N_RES,
                                    connectivity=CONNECTIVITY, seed=99)
    print(f"{method:<10} dense draw: "
          f"{'identical' if torch.equal(dense_a, dense_b) else 'differs':<9}   "
          f"as used (p_c={CONNECTIVITY}): "
          f"{'identical' if torch.equal(sparse_a, sparse_b) else 'differs'}")

gaussian   dense draw: differs     as used (p_c=0.1): differs
lcg        dense draw: differs     as used (p_c=0.1): differs
lfsr       dense draw: differs     as used (p_c=0.1): differs
sobol      dense draw: identical   as used (p_c=0.1): differs
halton     dense draw: identical   as used (p_c=0.1): differs


## 3. Stage 1: frozen reservoir, five initializers

Each method is injected into the frozen substrate and only the readout is
trained, at $k = T$. The stochastic methods are repeated over five seeds; the
deterministic ones are run once, as established above.

Results are appended to disk as each condition completes, so an interrupted run
resumes rather than restarting.

In [4]:
stage1 = {}

for name, (train_ds, test_ds) in splits.items():
    config = load_config(f"config_lsm_{name}")
    if config is None:
        raise RuntimeError(f"Run notebook 03 first — no tuned configuration for {name}.")

    conditions = [
        dict(kind="lsm", k=train_ds.seq_len, init_method=method, seed=seed,
             config=dict(config))
        for method in INIT_METHODS
        for seed in seeds_for(method, N_SEEDS)
    ]
    stage1[name] = run_grid(f"stage1_init_{name}", conditions, train_ds, test_ds)

stage1_init_ECG5000: 25 conditions to run


stage1_init_ECG5000:   0%|          | 0/25 [00:00<?, ?run/s]

stage1_init_Wafer: 25 conditions to run


stage1_init_Wafer:   0%|          | 0/25 [00:00<?, ?run/s]

stage1_init_InsectSound: 25 conditions to run


stage1_init_InsectSound:   0%|          | 0/25 [00:00<?, ?run/s]

In [5]:
tables = {}
for name, frame in stage1.items():
    table = (frame.groupby("init_method")
             .agg(macro_f1=("macro_f1", "mean"), sd=("macro_f1", "std"),
                  accuracy=("accuracy", "mean"), runs=("macro_f1", "size"),
                  firing_rate=("firing_rate", "mean"), sops=("sops", "mean"))
             .reindex(INIT_METHODS))
    table["sd"] = table["sd"].fillna(0.0)      # a single deterministic run
    table["vs_gaussian"] = table.macro_f1 - table.loc["gaussian", "macro_f1"]
    tables[name] = table
    print(f"\n{name} — frozen reservoir, k = T")
    display(table.round(4))


ECG5000 — frozen reservoir, k = T


,macro_f1,sd,accuracy,runs,firing_rate,sops,vs_gaussian
init_method,,,,,,,
gaussian,0.9496,0.0079,0.9512,5,0.3587,328935.1000,0.0000
lcg,0.9185,0.0607,0.9222,5,0.3584,329195.9188,-0.0312
lfsr,0.9310,0.0214,0.9332,5,0.3783,346041.6562,-0.0186
sobol,0.9406,0.0142,0.9428,5,0.3586,328035.7375,-0.0091
halton,0.9490,0.0204,0.9506,5,0.3561,326052.7938,-0.0006



Wafer — frozen reservoir, k = T


,macro_f1,sd,accuracy,runs,firing_rate,sops,vs_gaussian
init_method,,,,,,,
gaussian,0.8929,0.0163,0.9600,5,0.3707,369196.9312,0.0000
lcg,0.8363,0.1122,0.9419,5,0.3755,374164.0875,-0.0566
lfsr,0.8478,0.0784,0.9465,5,0.3923,389838.1062,-0.0451
sobol,0.8884,0.0283,0.9581,5,0.3740,371906.6188,-0.0045
halton,0.8556,0.0766,0.9523,5,0.3710,368782.9312,-0.0373



InsectSound — frozen reservoir, k = T


,macro_f1,sd,accuracy,runs,firing_rate,sops,vs_gaussian
init_method,,,,,,,
gaussian,0.3889,0.0064,0.4016,5,0.0822,322433.6500,0.0000
lcg,0.3893,0.0150,0.4000,5,0.0898,353096.4000,0.0004
lfsr,0.3801,0.0081,0.3899,5,0.0951,373136.2938,-0.0088
sobol,0.3663,0.0216,0.3751,5,0.0882,347354.6062,-0.0226
halton,0.3758,0.0123,0.3895,5,0.0892,349430.8562,-0.0131


The `vs_gaussian` column is the answer to the first research question, stated per
dataset: the performance given up, or gained, by abandoning the Gaussian
transform. It should be read against the `sd` column, since a deficit smaller
than the seed-to-seed spread is not evidence of a difference between methods.

The `firing_rate` and `sops` columns matter independently of accuracy. Two
initializers that classify equally well but drive the reservoir at different
rates do not cost the same to run, and on an edge device that difference is as
consequential as the accuracy.

The table states the answer; the figure below states its *precision*, which is
the part a table cannot convey. Each method is drawn as its mean advantage over
the Gaussian baseline with an error bar of one standard deviation across seeds.
A method whose interval crosses the zero line is not distinguishable from the
Gaussian baseline on that dataset, and two methods whose intervals overlap are
not distinguishable from one another.

The panels share a horizontal scale so that they may be compared with each
other: an effect that is small on one dataset and large on another should look
that way. Per the reporting convention of this thesis the figure and the table
present the same quantity, so only one of them belongs in the results chapter.


In [ ]:
plot_initializer_effect(
    stage1, order=INIT_METHODS, sharex=True,
    title="Effect of the initializer on a frozen reservoir ($k = T$)",
    ylabel="$\\Delta$ Macro-F1 vs. gaussian",
    save_path=f"{FIG_DIR}/stage1_initializer_effect.png",
)


## 4. Ablation: removing the variance control

The comparison above equalises the weight variance across methods. This section
removes that correction, so that the uniform generators keep their natural
standard deviation of $1/\sqrt{3}$ while the Gaussian baseline retains $1$.

The purpose is to measure the size of the confound the control removes. If the
ranking of methods changes when the correction is withdrawn, then a study run
without it would have been measuring scale while claiming to measure
distribution shape.

In [6]:
ablation = {}

for name, (train_ds, test_ds) in splits.items():
    config = load_config(f"config_lsm_{name}")
    conditions = [
        dict(kind="lsm", k=train_ds.seq_len, init_method=method, seed=seed,
             config=dict(config), match_variance=False)
        for method in INIT_METHODS
        for seed in seeds_for(method, N_SEEDS)
    ]
    ablation[name] = run_grid(f"stage1_ablation_{name}", conditions,
                              train_ds, test_ds)

stage1_ablation_ECG5000: 25 conditions to run


stage1_ablation_ECG5000:   0%|          | 0/25 [00:00<?, ?run/s]

stage1_ablation_Wafer: 25 conditions to run


stage1_ablation_Wafer:   0%|          | 0/25 [00:00<?, ?run/s]

stage1_ablation_InsectSound: 25 conditions to run


stage1_ablation_InsectSound:   0%|          | 0/25 [00:00<?, ?run/s]

In [7]:
for name in splits:
    controlled = tables[name][["macro_f1"]].rename(columns={"macro_f1": "with control"})
    uncontrolled = (ablation[name].groupby("init_method").macro_f1.mean()
                    .reindex(INIT_METHODS).rename("without control"))
    comparison = controlled.join(uncontrolled)
    comparison["difference"] = comparison["without control"] - comparison["with control"]
    print(f"\n{name} — effect of the variance control")
    display(comparison.round(4))


ECG5000 — effect of the variance control


,with control,without control,difference
init_method,,,
gaussian,0.9496,0.9496,0.0000
lcg,0.9185,0.9315,0.0130
lfsr,0.9310,0.9427,0.0117
sobol,0.9406,0.9298,-0.0108
halton,0.9490,0.9449,-0.0041



Wafer — effect of the variance control


,with control,without control,difference
init_method,,,
gaussian,0.8929,0.8929,0.0000
lcg,0.8363,0.8414,0.0051
lfsr,0.8478,0.8043,-0.0435
sobol,0.8884,0.8062,-0.0822
halton,0.8556,0.7082,-0.1474



InsectSound — effect of the variance control


,with control,without control,difference
init_method,,,
gaussian,0.3889,0.3889,0.0000
lcg,0.3893,0.3601,-0.0292
lfsr,0.3801,0.3553,-0.0248
sobol,0.3663,0.3637,-0.0026
halton,0.3758,0.3650,-0.0108


## 5. What this establishes

Stage 1 fixes the cost of cheap randomness on a frozen substrate, which is the
pessimistic case: the reservoir has no opportunity to compensate for whatever the
initializer got wrong. Whichever cheap method comes closest to the Gaussian
baseline here is carried into Stage 2, where truncated surrogate-gradient
training is allowed to adapt the reservoir and the question becomes how much of
the remaining deficit that training recovers.

The deterministic methods deserve separate comment in the results. Their zero
variance is not a small standard deviation but an exact one, and for an edge
deployment it is a property worth having in its own right: a device using a Sobol
or Halton reservoir needs neither a stored seed nor an entropy source, and
reproduces the identical network on every power-on.